In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\rawal\OneDrive\Desktop\mars\tickets_clean.csv')
print("Shape:", df.shape)
print(df['Priority_Level'].value_counts())
print("\nResolution time stats:")
print(df['Resolution_Time_Hours'].describe())
print("\nSatisfaction score stats:")
print(df['Satisfaction_Score'].describe())

Shape: (20000, 14)
Priority_Level
Low         7716
Medium      7570
High        3416
Critical    1298
Name: count, dtype: int64

Resolution time stats:
count    20000.000000
mean        39.230300
std         35.221884
min          1.000000
25%         11.000000
50%         27.000000
75%         58.000000
max        120.000000
Name: Resolution_Time_Hours, dtype: float64

Satisfaction score stats:
count    20000.000000
mean         3.723700
std          1.286989
min          1.000000
25%          3.000000
50%          4.000000
75%          5.000000
max          5.000000
Name: Satisfaction_Score, dtype: float64


In [20]:
# Use the ACTUAL mean RT per priority as boundaries
# Critical=12h, High=24h, Medium=44h, Low=45h
# Boundary between Critical and High = (12+24)/2 = 18h
# Boundary between High and Medium   = (24+44)/2 = 34h
# Boundary between Medium and Low    = (44+45)/2 = 44.5h

def rt_to_inferred(rt):
    if rt <= 18:
        return 4   # Critical
    elif rt <= 34:
        return 3   # High
    elif rt <= 44:
        return 2   # Medium
    else:
        return 1   # Low

df['inferred_severity_num'] = df['Resolution_Time_Hours'].apply(rt_to_inferred)

severity_map            = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Critical'}
df['inferred_severity'] = df['inferred_severity_num'].map(severity_map)

print("Inferred severity distribution:")
print(df['inferred_severity'].value_counts())

print("\nInferred vs Actual priority:")
print(pd.crosstab(df['Priority_Level'], df['inferred_severity']))

rt_acc = accuracy_score(df['priority_num'], df['inferred_severity_num'])
print(f"\nRT signal accuracy: {rt_acc:.2%}")

Inferred severity distribution:
inferred_severity
Critical    7559
Low         6705
High        3968
Medium      1768
Name: count, dtype: int64

Inferred vs Actual priority:
inferred_severity  Critical  High   Low  Medium
Priority_Level                                 
Critical               1028   203    28      39
High                   1784   757   550     325
Low                    2354  1514  3126     722
Medium                 2393  1494  3001     682

RT signal accuracy: 27.96%


In [21]:
df['severity_diff'] = abs(df['inferred_severity_num'] - df['priority_num'])

for threshold in [1, 2, 3]:
    rate  = (df['severity_diff'] >= threshold).mean()
    count = (df['severity_diff'] >= threshold).sum()
    print(f"Threshold >= {threshold}: {rate:.2%} mismatch ({count} tickets)")

Threshold >= 1: 72.04% mismatch (14407 tickets)
Threshold >= 2: 34.39% mismatch (6878 tickets)
Threshold >= 3: 11.91% mismatch (2382 tickets)


In [22]:
# Use threshold from Cell 3 that gives 20-35% mismatch
df['mismatch_label'] = (df['severity_diff'] >= 1).astype(int)
rate = df['mismatch_label'].mean()
print(f"Threshold 1 mismatch rate: {rate:.2%}")

# If still not in 20-35% range, we manually balance
if rate > 0.50:
    # Downsample mismatch class to get ~30% rate
    mismatch_df    = df[df['mismatch_label'] == 1]
    consistent_df  = df[df['mismatch_label'] == 0]
    
    # Keep all consistent, sample mismatches to get 30% ratio
    target_mismatch = int(len(consistent_df) * 0.43)  # 0.43 gives ~30% of total
    mismatch_sample = mismatch_df.sample(
        n=min(target_mismatch, len(mismatch_df)), 
        random_state=42
    )
    df_balanced = pd.concat([consistent_df, mismatch_sample]).sample(
        frac=1, random_state=42
    ).reset_index(drop=True)
    
    print(f"\nBalanced dataset:")
    print(df_balanced['mismatch_label'].value_counts())
    print(f"Mismatch rate: {df_balanced['mismatch_label'].mean():.2%}")
else:
    df_balanced = df.copy()
    print(f"\nNo balancing needed. Rate: {rate:.2%}")

Threshold 1 mismatch rate: 72.04%

Balanced dataset:
mismatch_label
0    5593
1    2404
Name: count, dtype: int64
Mismatch rate: 30.06%


In [23]:
import re

# Urgency score from text
URGENT_WORDS = [
    'urgent', 'asap', 'immediately', 'emergency', 'critical',
    'cannot access', 'not working', 'broken', 'error', 'failed',
    'crash', 'outage', 'down', 'blocked', 'fraud',
    'unauthorized', 'hacked', 'data loss', 'stolen', 'locked'
]

df_balanced['urgency_score'] = df_balanced['combined_text'].apply(
    lambda t: min(sum(1 for w in URGENT_WORDS if w in str(t).lower()), 10)
)

# Category severity
def category_severity(cat):
    cat = str(cat).strip()
    if cat == 'Fraud':        return 4
    elif cat == 'Technical':  return 3
    elif cat == 'Account':    return 3
    elif cat == 'Billing':    return 2
    else:                     return 1

df_balanced['cat_severity'] = df_balanced['Issue_Category'].apply(category_severity)

# Signal agreement: RT vs Category
agreement = (df_balanced['inferred_severity_num'] == df_balanced['cat_severity']).mean()
print(f"Signal Agreement (RT vs Category): {agreement:.2%}")
print(">> Save this for README")

print("\nFinal stats:")
print(f"Total tickets: {len(df_balanced)}")
print(f"Mismatch rate: {df_balanced['mismatch_label'].mean():.2%}")
print(f"Urgency score mean: {df_balanced['urgency_score'].mean():.2f}")

Signal Agreement (RT vs Category): 31.49%
>> Save this for README

Final stats:
Total tickets: 7997
Mismatch rate: 30.06%
Urgency score mean: 0.32


In [24]:
df_balanced.to_csv(
    r'C:\Users\rawal\OneDrive\Desktop\mars\tickets_labeled.csv',
    index=False
)
print("Saved!")
print(f"Shape: {df_balanced.shape}")
print(f"Columns: {df_balanced.columns.tolist()}")
print(f"\nMismatch distribution:")
print(df_balanced['mismatch_label'].value_counts())
print(f"Mismatch rate: {df_balanced['mismatch_label'].mean():.2%}")

Saved!
Shape: (7997, 23)
Columns: ['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject', 'Ticket_Description', 'Issue_Category', 'Priority_Level', 'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours', 'Assigned_Agent', 'Satisfaction_Score', 'combined_text', 'priority_num', 'rt_severity', 'text_severity', 'inferred_severity_raw', 'inferred_severity_num', 'inferred_severity', 'severity_diff', 'mismatch_label', 'urgency_score', 'cat_severity']

Mismatch distribution:
mismatch_label
0    5593
1    2404
Name: count, dtype: int64
Mismatch rate: 30.06%
